In [ ]:
# Listen to specific gold labels. Times in the label file are raw-audio;
# episode.audio() is trimmed, so clips are shifted by trim_start.
import json
from pathlib import Path

from drzavni_posao import MediaStore

AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
LABEL_DIR = Path("/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels")

EPISODE_KEY = "210"
# Zero-based indexes into gold["segments"]. Empty + empty STARTS = all labels
# at least MIN_SECONDS long.
INDEXES = []
# Raw-audio starts from an uncovered / segments table; matched within 50ms.
STARTS = [290.319, 302.290, 307.494]
MIN_SECONDS = 0.0
PAD_SECONDS = 0.25

label_path = LABEL_DIR / f"{EPISODE_KEY}.json"
if not label_path.exists():
    raise FileNotFoundError(f"No labels at {label_path}")

gold = json.loads(label_path.read_text(encoding="utf-8"))
trim_start = gold["proposal"]["audio"]["trim_start"]
media = MediaStore(AUDIO_PATHS, LABEL_DIR, **gold["proposal"]["audio"])
episode = media.episode(EPISODE_KEY)
audio = episode.audio()

labels = gold["segments"]


def resolve_starts(starts: list[float]) -> list[int]:
    resolved = []
    for start in starts:
        matches = [
            (abs(label["start"] - start), index)
            for index, label in enumerate(labels)
            if abs(label["start"] - start) < 0.05
        ]
        if not matches:
            raise ValueError(
                f"No gold label near start={start} in episode {EPISODE_KEY}"
            )
        resolved.append(min(matches)[1])
    return resolved


if INDEXES:
    queue_indexes = list(dict.fromkeys(INDEXES))
elif STARTS:
    queue_indexes = list(dict.fromkeys(resolve_starts(STARTS)))
else:
    queue_indexes = [
        index
        for index, label in enumerate(labels)
        if label["end"] - label["start"] >= MIN_SECONDS
    ]

for index in queue_indexes:
    if not 0 <= index < len(labels):
        raise IndexError(f"Invalid label index {index!r}")

queue = [
    {
        "index": index,
        "start": labels[index]["start"],
        "end": labels[index]["end"],
        "speaker": labels[index]["speaker"],
    }
    for index in queue_indexes
]

print(
    f"episode {EPISODE_KEY}: {len(queue)} clips to listen "
    f"({len(labels)} labeled total, trim_start={trim_start})"
)
for item in queue:
    print(
        f"  [{item['index']}] {item['start']:.3f}–{item['end']:.3f} "
        f"({item['end'] - item['start']:.3f}s)  {item['speaker']}"
    )


In [ ]:
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

from drzavni_posao.constants import SR

position = 0
output = widgets.Output()
prev_button = widgets.Button(description="Prev")
replay_button = widgets.Button(description="Replay")
next_button = widgets.Button(description="Next")
controls = [prev_button, replay_button, next_button]


def show():
    with output:
        clear_output(wait=True)
        if not queue:
            print("Nothing to listen to. Set INDEXES or STARTS and rerun prep.")
            for button in controls:
                button.disabled = True
            return
        item = queue[position]
        seconds = item["end"] - item["start"]
        print(
            f"{position + 1}/{len(queue)}  gold[{item['index']}]  "
            f"{item['start']:.3f}–{item['end']:.3f}s ({seconds:.3f}s)"
        )
        print(item["speaker"])
        start = max(0, int((item["start"] - trim_start - PAD_SECONDS) * SR))
        end = int((item["end"] - trim_start + PAD_SECONDS) * SR)
        display(Audio(audio[start:end], rate=SR, autoplay=True))


def step(delta):
    global position
    position = max(0, min(len(queue) - 1, position + delta))
    show()


prev_button.on_click(lambda _: step(-1))
replay_button.on_click(lambda _: show())
next_button.on_click(lambda _: step(1))
display(widgets.VBox([output, widgets.HBox(controls)]))
show()
